In [ ]:
short_names = {
    'Pòlissa/Póliza/Policy': 'POLICY',
    'Tecnologia/Tecnología/Technology': 'TECHNOLOGY',
    'Diàmetre comptador (cm)/Diámetro contador (cm)/Counter diameter (cm)': 'DIAMETER',
    'Ús/Uso/Use': 'USAGE',
    "Tipus d'habitatge/Tipo de vivienda/Type of housing": 'HOUSING',
    'Data/Fecha/Date': 'DATETIME',
    'Índex de lectura (L/h)/Índice de lectura (L/h)/Reading index (L/h)': 'CONSUMPTION',
}

#### Load data

In [ ]:
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import pandas as pd

file_path = '../data/lectures_horaries_ABD.parquet'

dataset = ds.dataset(file_path, format="parquet")
table = dataset.to_table()

In [ ]:
df = table.to_pandas()
df = df.rename(columns=short_names)
# Convert 'Data/Fecha/Date' column to datetime format
df['DATETIME'] = pd.to_datetime(df['DATETIME'])

df['DATE'] = df['DATETIME'].dt.date
df['HOUR'] = df['DATETIME'].dt.hour

df = df.drop(columns=['DATETIME'])

In [ ]:
df['FLOW'] = df.groupby('POLICY')['CONSUMPTION'].diff()
df = df.dropna(subset=["FLOW"])

In [ ]:
negative_policies = df[df['FLOW'] < 0]['POLICY'].values
outliers = df[(df['FLOW'] > 3000) & (df['USAGE'] == "AJUNTAMENT")]['POLICY'].values

filtered_df = df[~df['POLICY'].isin(negative_policies)]
filtered_df = filtered_df[~filtered_df['POLICY'].isin(outliers)]

#### Calculate the mean flow for each hour, and plot it

In [ ]:
# Assuming you have a DataFrame 'average_per_hour_usage' with columns 'HOUR', 'USAGE', and 'FLOW'
average_per_hour_usage = filtered_df.groupby(['USAGE', 'HOUR'])['FLOW'].mean().reset_index()
usage_types = average_per_hour_usage['USAGE'].unique()

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for i, usage in enumerate(usage_types):
  data = average_per_hour_usage[average_per_hour_usage['USAGE'] == usage]
  ax = axs[i // 2, i % 2]  # Assign subplot to each usage type
  ax.bar(data['HOUR'], data['FLOW'])  # Use bar function for bar graph
  ax.set_xlabel('Hour')
  ax.set_ylabel('Flow')
  ax.set_title(f'Flow for {usage}')

plt.tight_layout()
plt.show()